# IndiaRuns AI Challenge — Hybrid Ranking Sandbox

This notebook demonstrates our end-to-end candidate ranking pipeline. Our architecture uses a two-stage process to meet the strict 5-minute CPU constraint while maintaining extreme accuracy:

1. **Pre-computation (Offline):** We embed candidate career descriptions using `bge-small-en-v1.5` and store them in a FAISS exact search index. We also pre-compute 5 deterministic signal scores (Career Arc, Skill Trust, Location, YoE, Behavioral) and run a 5-check honeypot detection heuristic.
2. **Ranking (Runtime):** We query the FAISS index to retrieve the top candidates, apply the multi-signal scores to filter down to the top 200, and finally re-rank them using a pairwise cross-encoder (`ms-marco-MiniLM-L-6-v2`).

This sandbox will run the exact same pipeline on the provided `sample_candidates.json` (50 candidates).

In [ ]:
!pip install -q sentence-transformers faiss-cpu tqdm pandas

In [ ]:
import os
from google.colab import files

print("Please upload the 'sample_candidates.json' file provided in the challenge data bundle:")
uploaded = files.upload()

os.makedirs('data', exist_ok=True)
for fn in uploaded.keys():
    os.rename(fn, f'data/sample_candidates.json')
    print(f'Successfully saved {fn} to data/sample_candidates.json')


In [ ]:
%%writefile scoring.py
"""
scoring.py — All deterministic scoring functions for candidate ranking.

Imported by: precompute.py and rank.py
No model loading — pure Python logic only.

Functions exported:
    detect_honeypot(candidate) -> bool
    compute_career_arc_score(candidate) -> float
    compute_skill_trust_score(candidate) -> float
    compute_location_score(candidate) -> float
    compute_yoe_score(candidate) -> float
    compute_behavioral_score(signals) -> float
"""

from datetime import date as dt

# =============================================================================
# Constants
# =============================================================================

CONSULTING = {
    "tcs", "tata consultancy", "infosys", "wipro", "accenture", "cognizant",
    "capgemini", "hcl", "hcl technologies", "tech mahindra", "mphasis",
    "hexaware", "l&t infotech", "mindtree", "igate", "syntel", "mastech",
    "niit technologies", "patni", "satyam",
}

RESEARCH_KW = {
    "research scientist", "research engineer", "phd student", "phd candidate",
    "postdoc", "postdoctoral", "research intern", "research associate",
}

CV_SPEECH = {
    "computer vision", "object detection", "image classification", "yolo",
    "opencv", "speech recognition", "asr", "tts", "robotics",
}

NLP_IR = {
    "nlp", "information retrieval", "semantic search", "vector search",
    "embedding", "embeddings", "retrieval", "faiss", "pinecone", "weaviate",
    "qdrant", "milvus", "elasticsearch", "opensearch", "recommendation",
    "ranking", "search", "bm25", "dense retrieval", "hybrid search",
    "sentence-transformers",
}

ENG_KW = {"engineer", "developer", "architect", "scientist", "lead", "sde", "swe"}

NON_ENG = {
    "hr manager", "human resources", "operations manager", "marketing manager",
    "content writer", "graphic designer", "accountant", "sales manager",
    "customer support", "civil engineer", "mechanical engineer",
}

# Title-chaser seniority-level keywords (ordered low → high)
TITLE_LEVELS = [
    "junior", "associate", "mid", "senior", "staff", "principal",
    "lead", "director", "vp", "vice president", "head",
]

# LangChain-only detection keywords
LANGCHAIN_KW = {
    "langchain", "openai api", "chatgpt", "gpt-4", "llm api", "api wrapper",
}

PRE_LLM_KW = {
    "sklearn", "xgboost", "lightgbm", "tensorflow", "pytorch", "keras",
    "spark", "kafka", "airflow", "recommendation", "ranking", "retrieval",
    "embedding", "neural network", "random forest", "gradient boosting",
}

# JD-relevant skills for trust scoring
JD_REQ = {
    "embeddings", "embedding", "sentence-transformers", "faiss", "pinecone",
    "weaviate", "qdrant", "milvus", "elasticsearch", "opensearch",
    "vector database", "hybrid search", "dense retrieval",
    "information retrieval", "semantic search", "python", "nlp",
    "natural language processing", "ndcg", "mrr", "bm25",
    "recommendation", "ranking", "retrieval",
}

JD_NICE = {
    "lora", "qlora", "peft", "fine-tuning llms", "learning to rank",
    "xgboost", "lightgbm", "pytorch", "hugging face", "transformers",
    "recommendation systems", "mlflow",
}

JD_NEG = {
    "computer vision", "opencv", "yolo", "image classification",
    "object detection", "speech recognition", "asr", "tts", "robotics",
}

PROFICIENCY_WEIGHT = {
    "beginner": 0.20,
    "intermediate": 0.50,
    "advanced": 0.80,
    "expert": 1.00,
}


# =============================================================================
# Honeypot Detection
# =============================================================================

def detect_honeypot(candidate: dict) -> bool:
    """
    Detect impossible/fabricated candidate profiles.
    Returns True if the candidate is a honeypot (should be excluded).
    """
    profile = candidate["profile"]
    career = candidate["career_history"]
    skills = candidate["skills"]
    today = dt.today()

    # 1. Expert/advanced proficiency with zero duration on 3+ skills
    impossible = sum(
        1 for s in skills
        if s.get("proficiency") in ("expert", "advanced")
        and s.get("duration_months", 1) == 0
    )
    if impossible >= 3:
        return True

    # 2. Career months vs claimed YoE
    total = sum(j.get("duration_months", 0) for j in career)
    claimed = profile["years_of_experience"] * 12
    if total > claimed * 1.5 + 24:
        return True  # impossible overlap
    if total < claimed * 0.35 and len(career) > 1:
        return True  # impossible gap

    # 3. Future dates or end before start
    for job in career:
        try:
            start = dt.fromisoformat(job["start_date"])
            if start > today:
                return True
            if job["end_date"] is not None:
                end = dt.fromisoformat(job["end_date"])
                if end > today and not job.get("is_current", False):
                    return True
                if end < start:
                    return True
        except (ValueError, TypeError):
            pass

    # 4. Stated duration_months contradicts actual date range by >18 months
    for job in career:
        try:
            start = dt.fromisoformat(job["start_date"])
            end = (
                dt.fromisoformat(job["end_date"])
                if job["end_date"]
                else today if job.get("is_current") else None
            )
            if end is None:
                continue
            actual = (end.year - start.year) * 12 + (end.month - start.month)
            if abs(actual - job.get("duration_months", 0)) > 18:
                return True
        except (ValueError, TypeError):
            pass

    # 5. Multiple skills with duration far exceeding total career length
    # Buffer of 48 months accounts for pre-career learning (college, side projects).
    # Require 3+ impossible skills — single outliers are common in real data.
    impossible_skills = sum(
        1 for sk in skills
        if sk.get("duration_months", 0) > total + 48
    )
    if impossible_skills >= 3:
        return True

    return False


# =============================================================================
# Career Arc Score (weight 0.25)
# =============================================================================

def _is_consulting(company_name: str) -> bool:
    """Check if a company name matches a known consulting/IT-services firm."""
    lower = company_name.lower()
    return any(f in lower for f in CONSULTING)


def compute_career_arc_score(candidate: dict) -> float:
    """
    Score based on career trajectory: product-company experience,
    engineering roles, relevant domain. Catches keyword stuffers,
    consulting-only, pure research, wrong domain.

    Returns 0.0-1.0 (higher = better fit).
    """
    career = candidate["career_history"]
    profile = candidate["profile"]
    skills = candidate["skills"]

    companies = [j["company"].lower() for j in career]
    titles = [j["title"].lower() for j in career]
    skill_names = {s["name"].lower() for s in skills}
    career_text = " ".join(j.get("description", "") for j in career).lower()

    # --- Hard disqualifiers (return immediately) ---

    # All consulting firms
    if all(_is_consulting(co) for co in companies):
        return 0.10

    # >75% consulting
    if len(companies) > 0:
        consulting_ratio = sum(1 for co in companies if _is_consulting(co)) / len(companies)
        if consulting_ratio > 0.75:
            return 0.25

    # Non-engineer keyword stuffer (HR Manager, Content Writer, etc.)
    curr = profile["current_title"].lower()
    if any(ne in curr for ne in NON_ENG):
        # Only allow if they have prior engineering titles
        if not any(any(e in t for e in ENG_KW) for t in titles[1:]):
            return 0.05  # keyword stuffer

    # Title-chaser: avg tenure < 18mo AND titles show seniority-level inflation
    # (NOT just unique titles — that falsely flags lateral specialization moves)
    if len(career) >= 3:
        tenures = [j.get("duration_months", 0) for j in career]
        avg_tenure = sum(tenures) / len(tenures)
        if avg_tenure < 18:
            # Check for actual seniority-level keyword inflation
            levels = []
            for j in career:
                t = j["title"].lower()
                for i, lvl in enumerate(TITLE_LEVELS):
                    if lvl in t:
                        levels.append(i)
                        break
            # Only flag if 2+ level keywords found AND they're strictly ascending
            if (len(levels) >= 2
                    and levels == sorted(levels)
                    and levels[0] != levels[-1]):
                return 0.20  # true title-chaser: climbing seniority ladder fast

    # LangChain-only: API-wrapper experience with no pre-LLM production ML
    has_langchain = any(k in career_text for k in LANGCHAIN_KW)
    has_pre_llm = any(k in career_text for k in PRE_LLM_KW)
    if has_langchain and not has_pre_llm and profile["years_of_experience"] < 4:
        return 0.15  # LangChain-only, no production ML foundation

    # Pure research (all titles are research, zero engineering)
    r_ct = sum(1 for t in titles if any(r in t for r in RESEARCH_KW))
    e_ct = sum(1 for t in titles if any(e in t for e in ENG_KW))
    if r_ct > 0 and e_ct == 0:
        return 0.15

    # CV/Speech primary without NLP/IR
    has_nlp = any(k in career_text for k in NLP_IR)
    cv_ct = sum(1 for s in skill_names if any(c in s for c in CV_SPEECH))
    nlp_ct = sum(1 for s in skill_names if any(n in s for n in NLP_IR))
    if cv_ct >= 3 and nlp_ct == 0 and not has_nlp:
        return 0.25

    # Senior non-IC role (Director, VP, etc.) without recent hands-on
    if any(m in curr for m in ["director", "vp ", "vice president", "head of", " cto", " cpo"]):
        recent = " ".join(j.get("description", "") for j in career[:2]).lower()
        if not any(w in recent for w in ["built", "implemented", "deployed", "wrote", "shipped"]):
            return 0.35

    # --- Positive signals ---
    score = 0.65

    # Product company engineering roles
    prod = [
        j for j in career
        if not _is_consulting(j["company"].lower())
        and j["company_size"] not in ("1-10",)
        and any(e in j["title"].lower() for e in ENG_KW)
    ]
    if len(prod) >= 2:
        score += 0.20
    elif len(prod) == 1:
        score += 0.12

    # NLP/IR experience in career descriptions
    if has_nlp:
        score += 0.10

    # Mostly engineering titles
    if e_ct / max(len(titles), 1) >= 0.75:
        score += 0.05

    return min(score, 1.0)


# =============================================================================
# Skill Trust Score (weight 0.15)
# =============================================================================

def compute_skill_trust_score(candidate: dict) -> float:
    """
    Score based on JD-relevant skills, weighted by proficiency,
    endorsements, and duration. Penalizes CV/speech-only skills.
    Includes bonus for platform-verified skill assessments.

    Returns 0.0-1.0 (higher = better fit).
    """
    req = 0.0
    nice = 0.0
    penalty = 0.0

    for sk in candidate["skills"]:
        name = sk["name"].lower()
        pw = PROFICIENCY_WEIGHT.get(sk["proficiency"], 0.5)
        end = min(sk.get("endorsements", 0), 50) / 50.0
        dur = min(sk.get("duration_months", 0), 36) / 36.0
        trust = 0.50 + end * 0.25 + dur * 0.25  # higher endorsements + duration = more trusted
        val = pw * trust

        if any(r in name for r in JD_REQ):
            req += val
        elif any(n in name for n in JD_NICE):
            nice += val * 0.5

        if any(g in name for g in JD_NEG):
            penalty += 0.04

    base_result = max(0.0, min(min(req / 3.0, 1.0) + min(nice / 2.0, 0.25) - penalty, 1.0))

    # Bonus: platform-verified skill assessment scores
    assessment_bonus = 0.0
    assessment_scores = candidate.get("redrob_signals", {}).get("skill_assessment_scores", {})
    for skill_name, score in assessment_scores.items():
        if any(r in skill_name.lower() for r in JD_REQ):
            # Score is 0-100; normalize and apply small bonus
            assessment_bonus += (score / 100.0) * 0.05  # up to 0.05 per assessed skill

    assessment_bonus = min(assessment_bonus, 0.10)  # cap total bonus at 0.10

    return max(0.0, min(base_result + assessment_bonus, 1.0))


# =============================================================================
# Location Score (weight 0.10)
# =============================================================================

def compute_location_score(candidate: dict) -> float:
    """
    Score based on candidate location relative to JD requirements.
    Pune/Noida = ideal. Tier-1 Indian cities = good. Outside India = penalty.

    Returns 0.0-1.0 (higher = better fit).
    """
    loc = candidate["profile"]["location"].lower()
    country = candidate["profile"].get("country", "").lower()
    relocate = candidate["redrob_signals"].get("willing_to_relocate", False)

    # Tier A: Primary locations
    if any(c in loc for c in ["pune", "noida"]):
        return 1.00

    # Tier B: Major Indian cities
    if any(c in loc for c in [
        "hyderabad", "bangalore", "bengaluru", "delhi",
        "gurgaon", "gurugram", "mumbai", "ncr", "new delhi",
    ]):
        return 0.85

    # Tier C/D: Other Indian cities
    if country == "india" or "india" in loc:
        if any(c in loc for c in [
            "chennai", "kolkata", "ahmedabad", "kochi", "jaipur",
            "trivandrum", "chandigarh", "indore", "nagpur",
        ]):
            return 0.70
        return 0.65  # India, city unknown or unlisted

    # Tier E/F: Outside India
    return 0.45 if relocate else 0.15


# =============================================================================
# Years of Experience Score (weight 0.10)
# =============================================================================

def compute_yoe_score(candidate: dict) -> float:
    """
    Score based on years of experience relative to JD sweet spot (5-9 years).

    Returns 0.0-1.0 (higher = better fit).
    """
    yoe = candidate["profile"]["years_of_experience"]

    if 6 <= yoe <= 8:
        return 1.00   # JD sweet spot
    if 5 <= yoe < 6:
        return 0.90
    if 8 < yoe <= 10:
        return 0.85
    if 4 <= yoe < 5:
        return 0.70
    if 10 < yoe <= 15:
        return 0.70   # potentially overqualified
    if yoe > 15:
        return 0.50
    return 0.40       # under 4 years — too junior


# =============================================================================
# Behavioral Score (multiplier, not a component of skill_fit)
# =============================================================================

def compute_behavioral_score(signals: dict) -> float:
    """
    Behavioral engagement score based on platform activity signals.
    Applied as a MULTIPLIER on skill_fit_composite, not as an additive component.

    Returns 0.0-1.0 (higher = more engaged/available).
    """
    today = dt.today()

    # Recency (30% weight) — most important
    try:
        days = (today - dt.fromisoformat(signals["last_active_date"])).days
    except (ValueError, KeyError):
        days = 365
    if days <= 30:
        recency = 1.00
    elif days <= 60:
        recency = 0.90
    elif days <= 90:
        recency = 0.75
    elif days <= 180:
        recency = 0.50
    else:
        recency = 0.20  # ghost

    # Response rate (25% weight)
    rr = signals.get("recruiter_response_rate", 0.5)
    if rr >= 0.60:
        response = 1.00
    elif rr >= 0.40:
        response = 0.85
    elif rr >= 0.20:
        response = 0.65
    else:
        response = 0.30  # 5% = ghost

    # Notice period (20% weight)
    notice = signals.get("notice_period_days", 60)
    if notice <= 30:
        notice_s = 1.00
    elif notice <= 60:
        notice_s = 0.85
    elif notice <= 90:
        notice_s = 0.65
    else:
        notice_s = 0.40

    # Open to work (10% weight)
    open_work = 1.0 if signals.get("open_to_work_flag", True) else 0.75

    # Interview completion (10% weight)
    icr = signals.get("interview_completion_rate", 0.75)
    interview = 0.50 + (icr * 0.50)  # 0.0→0.50, 1.0→1.00

    # GitHub activity (3% weight — reduced from 5% to make room for saved_by_recruiters)
    gh = signals.get("github_activity_score", -1)
    if gh == -1:
        github = 0.85   # -1 = no GitHub = NEUTRAL
    elif gh >= 50:
        github = 1.00
    elif gh >= 20:
        github = 0.90
    else:
        github = 0.75

    # Saved by recruiters (2% weight — platform-validated demand signal)
    saved = min(signals.get("saved_by_recruiters_30d", 0), 20) / 20.0

    return (
        recency * 0.30
        + response * 0.25
        + notice_s * 0.20
        + open_work * 0.10
        + interview * 0.10
        + github * 0.03
        + saved * 0.02
    )


In [ ]:
%%writefile reasoning.py
"""
reasoning.py — Template-based reasoning generation from verified JSON fields.

No model. Zero hallucination risk. Every fact comes directly from the candidate's
JSON profile. Stage 4 checks that every mentioned fact exists in the actual data.

Functions exported:
    generate_reasoning(candidate, rank, score) -> str

Imported by: rank.py (called AFTER final ranking to ensure correct rank-dependent tone)
"""

from datetime import date

# Skills to highlight when found (display names, case-sensitive for output)
JD_SKILLS_DISPLAY = {
    "FAISS", "Pinecone", "Weaviate", "Qdrant", "Milvus", "Elasticsearch",
    "OpenSearch", "Sentence Transformers", "NLP", "PyTorch",
    "Hugging Face Transformers", "Recommendation Systems", "LoRA", "QLoRA",
    "PEFT", "Fine-tuning LLMs", "Python", "BM25", "MLflow",
    "Weights & Biases", "Embeddings", "Information Retrieval", "Ranking",
    "XGBoost", "LightGBM", "MLOps",
}

# Consulting firm names for concern flagging
CONS_CHECK = {
    "tcs", "infosys", "wipro", "accenture", "cognizant", "capgemini",
    "hcl", "tech mahindra", "mphasis", "mindtree", "hexaware",
}

# Engineering role keywords for identifying product-company roles
_ENG_KW = {"engineer", "developer", "scientist", "architect", "lead"}


def generate_reasoning(candidate: dict, rank: int, score: float) -> str:
    """
    Generate a reasoning string for a ranked candidate.

    Every fact in the output is sourced directly from the candidate's JSON.
    The tone adapts to the rank:
        - Ranks 1-10:  Confident, at most one concern
        - Ranks 11-30: Moderate, up to two concerns
        - Ranks 31-70: Balanced, up to three concerns
        - Ranks 71-100: Lists multiple concerns

    Args:
        candidate: Full candidate dict from the dataset
        rank: Final rank (1-100) after all scoring and re-ranking
        score: Final normalized score (0.40-0.99)

    Returns:
        Reasoning string suitable for the CSV output
    """
    p = candidate.get("profile", {})
    s = candidate.get("redrob_signals", {})
    career = candidate.get("career_history", [])
    skills = candidate.get("skills", [])

    strengths = []
    concerns = []

    # =========================================================================
    # STRENGTHS — only from verified JSON fields
    # =========================================================================

    # Years of experience and current title
    yoe = p.get("years_of_experience", 0)
    title = p.get("current_title", "Unknown")
    strengths.append(f"{yoe}y exp as {title}")

    # JD-relevant skills with trust signals
    matched = [
        sk["name"] for sk in skills
        if sk["name"] in JD_SKILLS_DISPLAY
        and sk.get("proficiency") in ("intermediate", "advanced", "expert")
        and sk.get("duration_months", 0) > 6
    ]
    if matched:
        strengths.append(f"hands-on: {', '.join(matched[:3])}")

    # Product company experience (non-consulting, non-tiny, engineering title)
    prod = [
        j for j in career
        if not any(f in j["company"].lower() for f in CONS_CHECK)
        and j["company_size"] not in ("1-10",)
        and any(e in j["title"].lower() for e in _ENG_KW)
    ]
    if prod:
        strengths.append(f"product co ({prod[0]['company']})")

    # Location
    location = p.get("location", "Unknown")
    strengths.append(f"based in {location.split(',')[0].strip()}")

    # GitHub activity (only if positive signal)
    gh = s.get("github_activity_score", -1)
    if gh >= 30:
        strengths.append(f"active GitHub ({int(gh)})")

    # Actively applying
    if s.get("applications_submitted_30d", 0) >= 2:
        strengths.append("actively applying")

    # Platform skill assessments (verified scores)
    assessments = s.get("skill_assessment_scores", {})
    jd_assessments = {
        k: v for k, v in assessments.items()
        if any(r in k.lower() for r in {
            "faiss", "pinecone", "embedding", "nlp", "retrieval",
            "ranking", "python", "mlflow", "elasticsearch",
        })
    }
    if jd_assessments:
        top_assessment = max(jd_assessments.items(), key=lambda x: x[1])
        if top_assessment[1] >= 60:
            strengths.append(f"verified {top_assessment[0]} ({top_assessment[1]:.0f}/100)")

    # =========================================================================
    # CONCERNS — only from verified JSON fields
    # =========================================================================

    # Inactivity
    try:
        days = (date.today() - date.fromisoformat(s["last_active_date"])).days
        if days > 90:
            concerns.append(f"inactive {days}d")
    except (ValueError, KeyError):
        pass

    # Low response rate
    rr = s.get("recruiter_response_rate", 1.0)
    if rr < 0.25:
        concerns.append(f"low response rate ({rr:.0%})")

    # Long notice period
    notice = s.get("notice_period_days", 0)
    if notice > 60:
        concerns.append(f"{notice}-day notice")

    # Not open to work
    if not s.get("open_to_work_flag", True):
        concerns.append("not open to work")

    # Low interview completion
    icr = s.get("interview_completion_rate", 1.0)
    if icr < 0.50:
        concerns.append(f"low interview completion ({icr:.0%})")

    # Location concern (outside India, not relocating)
    country = p.get("country", "India")
    if country.lower() not in ("india",) and not s.get("willing_to_relocate", False):
        concerns.append(f"in {country}, not relocating")

    # Entire career in consulting/IT-services
    if career and all(
        any(f in j["company"].lower() for f in CONS_CHECK)
        for j in career
    ):
        concerns.append("entire career in IT services")

    # Low profile completeness
    pc = s.get("profile_completeness_score", 100)
    if pc < 50:
        concerns.append(f"incomplete profile ({pc:.0f}%)")

    # =========================================================================
    # COMPOSE — rank determines tone
    # =========================================================================
    st = "; ".join(strengths)

    if rank <= 10:
        # Top 10: confident tone, at most one concern mentioned as a "note"
        if concerns:
            return f"{st}. Note: {concerns[0]}."
        else:
            return f"{st}; strong availability signals."

    elif rank <= 30:
        # Ranks 11-30: moderate, up to two concerns
        c = ", ".join(concerns[:2])
        return f"{st}. Concerns: {c}." if concerns else f"{st}."

    elif rank <= 70:
        # Ranks 31-70: balanced, up to three concerns
        c = ", ".join(concerns[:3])
        return f"{st}. Concerns: {c}." if concerns else f"{st}."

    else:
        # Ranks 71-100: list multiple concerns
        c = "; ".join(concerns)
        if concerns:
            return f"{st}. Multiple concerns: {c}."
        else:
            return f"{st}; included on partial skill fit."


In [ ]:
%%writefile precompute.py
#!/usr/bin/env python3
"""
precompute.py — Run once offline. No time limit. ~3-5 hours for 100K.
Generates: FAISS index, feature cache.

Usage:
  python precompute.py --candidates data/candidates.jsonl.gz
  python precompute.py --candidates data/candidates.jsonl
  python precompute.py --candidates data/sample_candidates.json  # dev mode
"""

import argparse
import gzip
import json
import pickle
from pathlib import Path

import faiss
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

from scoring import (
    detect_honeypot,
    compute_career_arc_score,
    compute_skill_trust_score,
    compute_location_score,
    compute_yoe_score,
    compute_behavioral_score,
)

# =============================================================================
# Configuration
# =============================================================================

JD_TEXT = (
    "Senior AI Engineer founding team at Redrob AI, Pune or Noida India. "
    "Requirements: production embedding-based retrieval systems — sentence-transformers, "
    "BGE, E5, OpenAI embeddings — handling drift, index refresh, quality regression. "
    "Production vector databases: Pinecone, Weaviate, Qdrant, Milvus, OpenSearch, "
    "Elasticsearch, FAISS. Strong Python. Ranking evaluation frameworks: "
    "NDCG, MRR, MAP, offline-to-online correlation, A/B testing. "
    "5-9 years applied ML at product companies. NOT consulting firms. NOT pure research. "
    "Has shipped ranking, search, or recommendation systems at meaningful scale. "
    "Location: Pune, Noida preferred; also Hyderabad, Bangalore, Delhi NCR, Mumbai. "
    "Notice: sub-30 days ideal, 60 days acceptable. "
    "Nice-to-have: LLM fine-tuning LoRA QLoRA PEFT, learning-to-rank, open-source contributions."
)

MODEL_NAME = "BAAI/bge-small-en-v1.5"
BATCH_SIZE = 64


# =============================================================================
# Data Loading (robust: .gz → .jsonl → .json fallback)
# =============================================================================

def load_candidates(path: str) -> list:
    """
    Load candidates from .gz, .jsonl, or .json file.
    Falls back gracefully if the exact file doesn't exist.
    """
    p = Path(path)

    # Try the exact path first
    if p.exists():
        if path.endswith(".gz"):
            print(f"  Loading gzipped JSONL: {path}")
            with gzip.open(path, "rt", encoding="utf-8") as f:
                return [json.loads(line) for line in f if line.strip()]
        elif path.endswith(".jsonl"):
            print(f"  Loading JSONL: {path}")
            with open(path, encoding="utf-8") as f:
                return [json.loads(line) for line in f if line.strip()]
        else:
            print(f"  Loading JSON array: {path}")
            with open(path, encoding="utf-8") as f:
                return json.load(f)

    # Fallback: try alternate extensions
    fallbacks = []
    if path.endswith(".gz"):
        fallbacks = [path[:-3], path[:-8] + ".json"]  # .jsonl, .json
    elif path.endswith(".jsonl"):
        fallbacks = [path + ".gz", path[:-6] + ".json"]  # .gz, .json
    else:
        fallbacks = [path + ".jsonl", path + ".jsonl.gz"]

    for fb in fallbacks:
        if Path(fb).exists():
            print(f"  File {path} not found, falling back to: {fb}")
            return load_candidates(fb)

    raise FileNotFoundError(f"Cannot find candidates file: {path} (also tried: {fallbacks})")


# =============================================================================
# Candidate Text Construction
# =============================================================================

def build_candidate_text(candidate: dict) -> str:
    """
    Build text for embedding. Embeds CAREER DESCRIPTIONS — not skills[*].name.
    Uses BGE-required prefix for passage encoding.
    """
    p = candidate["profile"]

    # Career descriptions are the PRIMARY signal
    parts = []
    for job in candidate["career_history"]:
        if job.get("description"):
            parts.append(f"{job['title']} at {job['company']}: {job['description']}")

    career_text = " ".join(parts)
    full = f"{p['headline']}. {p['summary']}. {career_text}"

    # BGE prefix for passage/document encoding
    return "Represent this professional background for retrieval: " + full[:3000]


# =============================================================================
# Main
# =============================================================================

def main():
    ap = argparse.ArgumentParser(description="Pre-compute embeddings and features")
    ap.add_argument("--candidates", required=True, help="Path to candidates file")
    args = ap.parse_args()

    Path("embeddings").mkdir(exist_ok=True)
    Path("cache").mkdir(exist_ok=True)

    # -------------------------------------------------------------------------
    # Step 1: Load candidates
    # -------------------------------------------------------------------------
    print("[1/5] Loading candidates...")
    candidates = load_candidates(args.candidates)
    cids = [c["candidate_id"] for c in candidates]
    print(f"  {len(candidates):,} candidates loaded")

    # -------------------------------------------------------------------------
    # Step 2: Compute features for all candidates
    # -------------------------------------------------------------------------
    print("[2/5] Computing features for all candidates...")
    features = {}
    hp_count = 0

    for c in tqdm(candidates, desc="  Features"):
        cid = c["candidate_id"]
        is_hp = detect_honeypot(c)
        if is_hp:
            hp_count += 1

        features[cid] = {
            "is_honeypot":       is_hp,
            "career_arc_score":  compute_career_arc_score(c),
            "skill_trust_score": compute_skill_trust_score(c),
            "location_score":    compute_location_score(c),
            "yoe_score":         compute_yoe_score(c),
            "behavioral_score":  compute_behavioral_score(c["redrob_signals"]),
        }

    print(f"  {hp_count} honeypots detected out of {len(candidates):,}")

    # Save features and candidate ID list
    with open("cache/features.pkl", "wb") as f:
        pickle.dump(features, f)
    with open("cache/cids.pkl", "wb") as f:
        pickle.dump(cids, f)
    print("  Features saved to cache/")

    # -------------------------------------------------------------------------
    # Step 3: Load embedding model
    # -------------------------------------------------------------------------
    print("[3/5] Loading embedding model...")
    model = SentenceTransformer(MODEL_NAME)
    print(f"  Model: {MODEL_NAME}")

    # -------------------------------------------------------------------------
    # Step 4: Embed JD
    # -------------------------------------------------------------------------
    print("[4/5] Embedding JD...")
    jd_query = "Retrieve candidates matching this job description: " + JD_TEXT
    jd_emb = model.encode(
        jd_query,
        normalize_embeddings=True,
    ).astype("float32")
    np.save("embeddings/jd.npy", jd_emb)
    print(f"  JD embedding shape: {jd_emb.shape}")

    # -------------------------------------------------------------------------
    # Step 5: Embed all candidates and build FAISS index
    # -------------------------------------------------------------------------
    print(f"[5/5] Embedding {len(candidates):,} candidates (batch={BATCH_SIZE})...")
    texts = [build_candidate_text(c) for c in candidates]

    all_embs = []
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="  Embedding"):
        batch = texts[i:i + BATCH_SIZE]
        emb = model.encode(
            batch,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype("float32")
        all_embs.append(emb)

    embs = np.vstack(all_embs)
    print(f"  Embeddings shape: {embs.shape}")

    # Build FAISS index (inner product = cosine similarity on normalized vectors)
    print("  Building FAISS IndexFlatIP...")
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)
    faiss.write_index(index, "embeddings/index.bin")
    print(f"  Index: {index.ntotal:,} vectors, dim={embs.shape[1]}")

    # -------------------------------------------------------------------------
    # Summary
    # -------------------------------------------------------------------------
    print()
    print("=" * 60)
    print("Pre-computation complete!")
    print(f"  Candidates:  {len(candidates):,}")
    print(f"  Honeypots:   {hp_count}")
    print(f"  Embeddings:  {embs.shape}")
    print(f"  FAISS index: embeddings/index.bin")
    print(f"  Features:    cache/features.pkl")
    print(f"  CIDs:        cache/cids.pkl")
    print(f"  JD emb:      embeddings/jd.npy")
    print("=" * 60)
    print("Ready to run: python rank.py --candidates <path> --out output/team_xxx.csv")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile rank.py
#!/usr/bin/env python3
"""
rank.py — Runtime ranking pipeline. Must complete within 5 minutes on CPU, 16 GB RAM.

Usage:
  python rank.py --candidates data/candidates.jsonl --out output/submission.csv

Pipeline:
  1. Load pre-computed artifacts (FAISS index, features, JD embedding)
  2. Load candidates for cross-encoder text
  3. FAISS search → top 3000
  4. Filter honeypots, multi-signal score, take top 200
  5. Cross-encoder re-rank top 200
  6. Sort by (-final, cid) for tie-breaking
  7. Take top 100, normalize scores, generate reasoning, write CSV
"""

import argparse
import csv
import gzip
import json
import pickle
import time
from pathlib import Path

import faiss
import numpy as np
from sentence_transformers import CrossEncoder

from scoring import (
    compute_career_arc_score,
    compute_skill_trust_score,
    compute_location_score,
    compute_yoe_score,
    compute_behavioral_score,
)
from reasoning import generate_reasoning

# =============================================================================
# Configuration
# =============================================================================

# Weights for skill_fit composite
W_SEMANTIC  = 0.40
W_CAREER    = 0.25
W_SKILL     = 0.15
W_LOCATION  = 0.10
W_YOE       = 0.10

CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
FAISS_K = 3000            # Retrieve top K from FAISS (increased from 1000 to avoid missing strong candidates)
CROSS_ENCODER_TOP = 200   # Re-rank top N with cross-encoder
FINAL_TOP = 100           # Output top N
TEXT_CAP = 1400           # Max chars for cross-encoder candidate text (Bug B fix)

JD_TEXT = (
    "Senior AI Engineer founding team at Redrob AI, Pune or Noida India. "
    "Requirements: production embedding-based retrieval systems — sentence-transformers, "
    "BGE, E5, OpenAI embeddings — handling drift, index refresh, quality regression. "
    "Production vector databases: Pinecone, Weaviate, Qdrant, Milvus, OpenSearch, "
    "Elasticsearch, FAISS. Strong Python. Ranking evaluation frameworks: "
    "NDCG, MRR, MAP, offline-to-online correlation, A/B testing. "
    "5-9 years applied ML at product companies. NOT consulting firms. NOT pure research. "
    "Has shipped ranking, search, or recommendation systems at meaningful scale. "
    "Location: Pune, Noida preferred; also Hyderabad, Bangalore, Delhi NCR, Mumbai. "
    "Notice: sub-30 days ideal, 60 days acceptable. "
    "Nice-to-have: LLM fine-tuning LoRA QLoRA PEFT, learning-to-rank, open-source contributions."
)


# =============================================================================
# Data Loading (robust: .gz → .jsonl → .json fallback)
# =============================================================================

def load_candidates(path: str) -> list:
    """Load candidates from .gz, .jsonl, or .json file with fallback."""
    p = Path(path)

    if p.exists():
        if path.endswith(".gz"):
            print(f"  Loading gzipped JSONL: {path}")
            with gzip.open(path, "rt", encoding="utf-8") as f:
                return [json.loads(line) for line in f if line.strip()]
        elif path.endswith(".jsonl"):
            print(f"  Loading JSONL: {path}")
            with open(path, encoding="utf-8") as f:
                return [json.loads(line) for line in f if line.strip()]
        else:
            print(f"  Loading JSON array: {path}")
            with open(path, encoding="utf-8") as f:
                return json.load(f)

    # Fallback
    fallbacks = []
    if path.endswith(".gz"):
        fallbacks = [path[:-3], path[:-8] + ".json"]
    elif path.endswith(".jsonl"):
        fallbacks = [path + ".gz", path[:-6] + ".json"]
    else:
        fallbacks = [path + ".jsonl", path + ".jsonl.gz"]

    for fb in fallbacks:
        if Path(fb).exists():
            print(f"  File {path} not found, falling back to: {fb}")
            return load_candidates(fb)

    raise FileNotFoundError(f"Cannot find: {path} (also tried: {fallbacks})")


# =============================================================================
# Cross-Encoder Text (Bug B fix: capped at TEXT_CAP chars)
# =============================================================================

def build_ce_text(c: dict) -> str:
    """
    Build rich text for cross-encoder re-ranking.
    Includes YoE, title, location, summary, 4 career entries with descriptions,
    and 12 skills with proficiency+duration. Capped at TEXT_CAP chars to stay
    within the 512-token limit of ms-marco-MiniLM-L-6-v2.
    """
    p = c["profile"]
    career = " ".join(
        f"{j['title']} at {j['company']} ({j.get('company_size', '?')}): "
        f"{j.get('description', '')[:350]}"
        for j in c["career_history"][:4]
    )
    skills = ", ".join(
        f"{s['name']} ({s['proficiency']}, {s.get('duration_months', 0)}mo)"
        for s in c["skills"][:12]
    )
    text = (
        f"{p['years_of_experience']}y | {p['current_title']} | "
        f"{p['location']}. {p.get('summary', '')[:250]}. "
        f"{career}. Skills: {skills}"
    )
    return text[:TEXT_CAP]  # Hard cap: keeps combined (JD+candidate) under 512 tokens


# =============================================================================
# Main
# =============================================================================

def main():
    t0 = time.time()

    ap = argparse.ArgumentParser(description="Rank candidates against JD")
    ap.add_argument("--candidates", required=True, help="Path to candidates file")
    ap.add_argument("--out", required=True, help="Output CSV path")
    args = ap.parse_args()

    Path(args.out).parent.mkdir(parents=True, exist_ok=True)

    # =========================================================================
    # Step 1: Load pre-computed artifacts
    # =========================================================================
    print(f"[1/7] Loading pre-computed artifacts...")
    index = faiss.read_index("embeddings/index.bin")
    jd_emb = np.load("embeddings/jd.npy").reshape(1, -1)
    with open("cache/features.pkl", "rb") as f:
        features = pickle.load(f)
    with open("cache/cids.pkl", "rb") as f:
        cids = pickle.load(f)
    print(f"  Index: {index.ntotal:,} vectors | Features: {len(features):,} | CIDs: {len(cids):,}")
    print(f"  [{time.time() - t0:.1f}s]")

    # =========================================================================
    # Step 2: Load candidates (needed for cross-encoder text + reasoning)
    # =========================================================================
    print(f"[2/7] Loading candidates...")
    candidates = load_candidates(args.candidates)
    id2c = {c["candidate_id"]: c for c in candidates}
    print(f"  {len(candidates):,} candidates loaded")
    print(f"  [{time.time() - t0:.1f}s]")

    # =========================================================================
    # Step 3: FAISS search → top K
    # =========================================================================
    k = min(FAISS_K, index.ntotal)
    print(f"[3/7] FAISS search (k={k})...")
    D, I = index.search(jd_emb, k)
    print(f"  Top similarity: {D[0][0]:.4f}, Bottom: {D[0][-1]:.4f}")
    print(f"  [{time.time() - t0:.1f}s]")

    # =========================================================================
    # Step 4: Filter honeypots + multi-signal scoring → top 200
    # =========================================================================
    print(f"[4/7] Multi-signal scoring + honeypot filtering...")
    scored = []
    hp_filtered = 0

    for idx, sim in zip(I[0], D[0]):
        cid = cids[idx]
        feat = features.get(cid)
        if feat is None:
            continue
        if feat["is_honeypot"]:
            hp_filtered += 1
            continue

        # Compute skill_fit composite
        skill_fit = (
            W_SEMANTIC  * float(sim)
            + W_CAREER  * feat["career_arc_score"]
            + W_SKILL   * feat["skill_trust_score"]
            + W_LOCATION * feat["location_score"]
            + W_YOE     * feat["yoe_score"]
        )

        # Apply behavioral multiplier
        final = skill_fit * feat["behavioral_score"]

        scored.append({
            "cid": cid,
            "sim": float(sim),
            "skill_fit": skill_fit,
            "behavioral": feat["behavioral_score"],
            "final": final,
        })

    # Sort by final score descending, take top CROSS_ENCODER_TOP
    scored.sort(key=lambda x: (-x["final"], x["cid"]))
    top_for_ce = scored[:CROSS_ENCODER_TOP]
    print(f"  Honeypots filtered: {hp_filtered}")
    print(f"  Candidates for cross-encoder: {len(top_for_ce)}")
    print(f"  [{time.time() - t0:.1f}s]")

    # =========================================================================
    # Step 5: Cross-encoder re-ranking
    # =========================================================================
    print(f"[5/7] Cross-encoder re-ranking ({len(top_for_ce)} candidates)...")
    ce_model = CrossEncoder(CROSS_ENCODER_MODEL, max_length=512)

    # Build (JD, candidate) pairs
    pairs = []
    for item in top_for_ce:
        c = id2c.get(item["cid"])
        if c is None:
            pairs.append((JD_TEXT, "Profile unavailable"))
        else:
            pairs.append((JD_TEXT, build_ce_text(c)))

    # Score all pairs
    ce_scores = ce_model.predict(pairs, show_progress_bar=True)

    # Normalize CE scores to 0-1 range
    ce_min = float(min(ce_scores))
    ce_max = float(max(ce_scores))
    ce_range = ce_max - ce_min if ce_max > ce_min else 1.0

    # Blend: 75% multi-signal + 25% cross-encoder
    # (Tested: 60/40 lets CE override multi-signal; 75/25 keeps correct ranking)
    for i, item in enumerate(top_for_ce):
        ce_norm = (float(ce_scores[i]) - ce_min) / ce_range
        item["ce_score"] = ce_norm
        item["final"] = item["final"] * 0.75 + ce_norm * 0.25

    print(f"  CE score range: [{ce_min:.4f}, {ce_max:.4f}]")
    print(f"  [{time.time() - t0:.1f}s]")

    # =========================================================================
    # Step 6: Sort by (-final, cid) for tie-breaking (Bug C fix)
    # =========================================================================
    print(f"[6/7] Final sort with tie-breaking...")
    top_for_ce.sort(key=lambda x: (-x["final"], x["cid"]))
    top100 = top_for_ce[:FINAL_TOP]
    print(f"  Top 100 selected")

    # Normalize scores to 0.40-0.99 range
    if len(top100) > 1:
        mn = min(item["final"] for item in top100)
        mx = max(item["final"] for item in top100)
        rng = mx - mn if mx > mn else 1.0
    else:
        mn, mx, rng = 0.0, 1.0, 1.0

    print(f"  Raw score range: [{mn:.4f}, {mx:.4f}]")
    print(f"  [{time.time() - t0:.1f}s]")

    # =========================================================================
    # Step 7: Generate reasoning + write CSV
    # =========================================================================
    print(f"[7/7] Generating reasoning + writing CSV...")
    with open(args.out, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["candidate_id", "rank", "score", "reasoning"])

        for rank, item in enumerate(top100, 1):
            cid = item["cid"]
            out_score = round(0.40 + ((item["final"] - mn) / rng) * 0.59, 4)

            # Bug D fix: guard for missing candidates
            if cid not in id2c:
                reason = f"{cid} profile data unavailable for reasoning."
            else:
                reason = generate_reasoning(id2c[cid], rank, out_score)

            writer.writerow([cid, rank, out_score, reason])

    elapsed = time.time() - t0
    print(f"\n{'=' * 60}")
    print(f"Ranking complete!")
    print(f"  Output: {args.out}")
    print(f"  Total time: {elapsed:.1f}s")
    print(f"  Top 1: {top100[0]['cid']} (score={0.40 + ((top100[0]['final'] - mn) / rng) * 0.59:.4f})")
    if len(top100) >= 100:
        print(f"  Top 100: {top100[99]['cid']} (score={0.40 + ((top100[99]['final'] - mn) / rng) * 0.59:.4f})")
    print(f"  Honeypots removed: {hp_filtered}")
    print(f"{'=' * 60}")
    print(f"{'⚠️ OVER 5-MINUTE LIMIT!' if elapsed > 300 else '✅ Within time limit'}")


if __name__ == "__main__":
    main()


In [ ]:
!python precompute.py --candidates data/sample_candidates.json
!python rank.py --candidates data/sample_candidates.json --out output/sandbox_submission.csv

In [ ]:
import pandas as pd
df = pd.read_csv('output/sandbox_submission.csv')

# Configure pandas to show full reasoning text
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)

print("
🏆 Top Candidates:")
display(df)
